In [2]:
!pip install feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 1.4 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=39cd3d104a9913934936b497c385cac5c1ec5cecb7997024c4be956507631ac6
  Stored in directory: /root/.cache/pip/wheels/3b/25/2a/105d6a15df6914f4d15047691c6c28f9052cc1173e40285d03
Successfully built sgmllib3k


In [4]:
!pip install pyshorteners

  Preparing metadata (setup.py) ... done
  Created wheel for pyshorteners: filename=pyshorteners-1.0.1-py3-none-any.whl size=17479 sha256=a8133709ee8a07dfd5c075d7a5ff89a8c290c27df654658d9054d88b17e162c0
  Stored in directory: /root/.cache/pip/wheels/40/25/54/000cc118ff192ee36c95b1374ee4c42d5d39143d940de5908a
Successfully built pyshorteners


In [5]:
import feedparser
from bs4 import BeautifulSoup
import re
import base64
import requests
import pyshorteners

def extract_real_url(google_url):
    """Extract the real news article URL from Google News RSS link if possible."""
    # Only decode if URL matches the /articles/ pattern
    match = re.search(r'/articles/([A-Za-z0-9_-]+)', google_url)
    if match:
        encoded_part = match.group(1)
        # Add padding if needed for Base64 decoding
        padding = len(encoded_part) % 4
        if padding:
            encoded_part += '=' * (4 - padding)
        try:
            decoded_bytes = base64.urlsafe_b64decode(encoded_part)
            decoded_text = decoded_bytes.decode('utf-8', errors='ignore')
            url_match = re.search(r'(https?://[^\s"\']+)', decoded_text)
            if url_match:
                return url_match.group(1)
        except Exception:
            pass  # If decoding fails, just return the original Google News link
    # If it's not /articles/ pattern or decoding fails, try to extract from redirect
    try:
        response = requests.head(google_url, timeout=5, allow_redirects=True)
        return response.url
    except Exception:
        return google_url

def shorten_url(long_url):
    """Shorten a URL using TinyURL."""
    try:
        shortener = pyshorteners.Shortener()
        return shortener.tinyurl.short(long_url)
    except Exception:
        return long_url

def fetch_city_news_rss(city, max_articles=10, time_period="12h"):
    """Fetch news articles for a specific city using Google News RSS feeds."""
    print(f"\nFetching RSS news for city: {city}")

    query = city.replace(" ", "+")
    rss_url = f"https://news.google.com/rss/search?q={query}+when:{time_period}&hl=en-IN&gl=IN&ceid=IN:en"

    feed = feedparser.parse(rss_url)
    if not feed.entries:
        print(f"No RSS entries found for {city}")
        return []

    articles = []
    for entry in feed.entries[:max_articles]:
        # Title and source
        title_parts = entry.title.split(" - ")
        title = title_parts[0].strip() if len(title_parts) > 1 else entry.title.strip()
        source = title_parts[-1].strip() if len(title_parts) > 1 else "Unknown"

        # Description
        raw_description = entry.get("description", "")
        soup = BeautifulSoup(raw_description, 'html.parser')
        clean_description = soup.get_text(separator=' ', strip=True) or "No description available"

        # Extract and shorten the real news URL
        google_news_link = entry.link
        real_url = extract_real_url(google_news_link)
        news_link = shorten_url(real_url)

        articles.append({
            "title": title,
            "description": clean_description[:200] + "..." if len(clean_description) > 200 else clean_description,
            "news_link": news_link,
            "source": source,
            "published_at": entry.get("published", "Unknown date")
        })

    print(f"Fetched {len(articles)} RSS articles for {city}")
    return articles

if __name__ == "__main__":
    city = input("Enter a city to get recent news: ").strip()
    articles = fetch_city_news_rss(city, max_articles=10, time_period="12h")

    print(f"\nNews for {city}:")
    for i, article in enumerate(articles, 1):
        print(f"\n{i}. {article['title']}")
        print(f"   Source: {article['source']}")
        print(f"   Published: {article['published_at']}")
        print(f"   Description: {article['description']}")
        print(f"   News Link: {article['news_link']}")


Enter a city to get recent news: Delhi

Fetching RSS news for city: Delhi
Fetched 10 RSS articles for Delhi

News for Delhi:

1. Delhi News Live Updates: IMD rules out heatwave conditions in Delhi for next 7 days; partly cloudy sky likely today
   Source: The Indian Express
   Published: Wed, 14 May 2025 13:19:04 GMT
   Description: Delhi News Live Updates: IMD rules out heatwave conditions in Delhi for next 7 days; partly cloudy sky likely today The Indian Express
   News Link: https://tinyurl.com/27t24kv2

2. Delhi airport issues fresh advisory amid India-Pak tensions
   Source: DD News
   Published: Wed, 14 May 2025 12:12:28 GMT
   Description: Delhi airport issues fresh advisory amid India-Pak tensions DD News
   News Link: https://tinyurl.com/22j55h78

3. IPL 2025: Why Jake Fraser-McGurk is not returning for Delhi Capitals; Mitchell Starc also uncertain
   Source: Times of India
   Published: Wed, 14 May 2025 10:44:00 GMT
   Description: IPL 2025: Why Jake Fraser-McGurk is not ret